# Define a location and analyse the time series of HOSTRADA climate variables

In [ ]:
import os
import html as html_lib
from datetime import date

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import requests
import ipywidgets as widgets
from IPython.display import display

from hostrada4py import hostradaPoint as hp
from hostrada4py import hostradaPointUI as hpui

os.environ["HOSTRADA_NETCDF_SUBSET_MODE"] = "full"
#os.environ["HOSTRADA_NETCDF_SUBSET_MODE"] = "auto"


## Interactive selection of the HOSTRADA value


In [ ]:
# Climate variables available in the original notebook.
CLIMATE_VARIABLES = {
    "Outside air temperature (tas)": "tas",
    "Wind speed (sfcWind)": "sfcWind",
    "Wind direction (sfcWind_direction)": "sfcWind_direction",
    "Urban Heat Island Intensity (uhi)": "uhi",
    "Global radiation (rsds)": "rsds",
    "Cloud cover (clt)": "clt",
    "Relative humidity (hurs)": "hurs",
    "Water vapor mixing ratio (mixr)": "mixr",
    "Dew point temperature (tdew)": "tdew",
}

# Default retained from the original notebook.
HOSTRADA_VAR = "tas"

climate_dropdown = widgets.Dropdown(
    options=[(label, value) for label, value in CLIMATE_VARIABLES.items()],
    value=HOSTRADA_VAR,
    description="Climate variable:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="560px"),
)
climate_status = widgets.HTML()


def _set_climate_variable(change=None):
    global HOSTRADA_VAR
    HOSTRADA_VAR = climate_dropdown.value
    selected_label = next(
        label for label, value in CLIMATE_VARIABLES.items() if value == HOSTRADA_VAR
    )
    climate_status.value = (
        f"<b>Selected:</b> {html_lib.escape(selected_label)} &nbsp; "
        f"(<code>HOSTRADA_VAR = '{HOSTRADA_VAR}'</code>)"
    )


climate_dropdown.observe(_set_climate_variable, names="value")
_set_climate_variable()
display(widgets.VBox([climate_dropdown, climate_status]))


## Interactive definition of the location and time period, and download of the HOSTRADA values


In [ ]:
point_ui = hpui.create_hostrada_point_ui(
    namespace=globals(),
    extractor=hp.extract_values_for_point,
    initial_lat=52.51712,
    initial_lon=13.32259,
    initial_address="Einsteinufer 43–53, 10587 Berlin",
    initial_start="2025-01-01T00:00",
    initial_end="2025-12-31T23:00",
    initial_output_directory=".",
    initial_output_filename=None,
)

display(point_ui.widget)


## Yearly time series of the HOSTRADA variable

In [ ]:
df = pd.read_csv(fn)
df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time")

fig, ax = plt.subplots(figsize=(12,5))
ax.plot(df["time"], df[HOSTRADA_VAR])
ax.set_xlabel("Datum")
ax.set_ylabel(HOSTRADA_VAR)

if HOSTRADA_VAR == "tas":
    title = "Air temperature in °C"
elif HOSTRADA_VAR == "uhi":
    title = "Urban Heat Island Intensity in °C"
elif HOSTRADA_VAR == "sfcWind":
    title = "Wind speed in m/s"
elif HOSTRADA_VAR == "sfcWind_direction":
    title = "Wind direction in degree"
elif HOSTRADA_VAR == "rsds":
    title = "Global radiation in W/m2"
elif HOSTRADA_VAR == "clt":
    title = "Cloud cover in eighth"
elif HOSTRADA_VAR == "hurs":
    title = "Relative humidity in percent"
elif HOSTRADA_VAR == "tdew":
    title = "Dew point temperature in °C"
elif HOSTRADA_VAR == "mixr":
    title = "Water vapor mixing ratio in g H20/kg dry air"
else:
    title = "unknown"

ax.set_title(title)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m %H:%M"))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())

plt.xticks(rotation=45)
plt.grid(True)
plt.show()


## Monthly mean values of the HOSTRADA variable

In [ ]:
df = pd.read_csv(fn)
df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time")
monthly_mean = df.resample("M", on="time")[HOSTRADA_VAR].mean()

plt.figure(figsize=(10,5))
plt.plot(monthly_mean.index, monthly_mean.values, marker="o")
plt.xlabel("Month")
plt.ylabel(HOSTRADA_VAR)

if HOSTRADA_VAR == "tas":
    title = "Monthly mean air temperature in °C"
elif HOSTRADA_VAR == "uhi":
    title = "Monthly mean Urban Heat Island Intensity in °C"
elif HOSTRADA_VAR == "sfcWind":
    title = "Monthly mean wind speed in m/s"
elif HOSTRADA_VAR == "sfcWind_direction":
    title = "Monthly mean wind direction in degree"
elif HOSTRADA_VAR == "rsds":
    title = "Monthly mean global radiotion in W/m2"
elif HOSTRADA_VAR == "clt":
    title = "Monthly mean cloud cover in eighth"
elif HOSTRADA_VAR == "hurs":
    title = "Monthly mean relative humidity in percent"
elif HOSTRADA_VAR == "tdew":
    title = "Monthly mean dew point temperature in °C"
elif HOSTRADA_VAR == "mixr":
    title = "Monthly water vapor mixing ratio in g H20/kg dry air"
else:
    title = "unknown"
    
plt.title(title)
plt.grid(True)
plt.show()


## Heatmap of the HOSTRADA variable

In [ ]:
df["dayofyear"] = df["time"].dt.dayofyear
df["hour"] = df["time"].dt.hour
heatmap_year = df.pivot_table(
    values=HOSTRADA_VAR,
    index="hour",
    columns="dayofyear")

plt.figure(figsize=(16,5))
plt.xlabel("Tag des Jahres")
plt.ylabel("Stunde")

if HOSTRADA_VAR == "tas":
    lable = "Air temperature (°C)"
elif HOSTRADA_VAR == "uhi":
    lable = "Urban Heat Island Intensity (°C)"
elif HOSTRADA_VAR == "sfcWind":
    lable = "Wind speed (m/s)"
elif HOSTRADA_VAR == "sfcWind_direction":
    lable = "Wind direction (degree)"
elif HOSTRADA_VAR == "rsds":
    lable = "Global radiation (W/m2)"
elif HOSTRADA_VAR == "clt":
    lable = "Cloud cover (eighth)"
elif HOSTRADA_VAR == "hurs":
    lable = "Relative humidity (percent)"
elif HOSTRADA_VAR == "tdew":
    lable = "Dew point temperature (°C)"
elif HOSTRADA_VAR == "mixr":
    lable = "Water vapor mixing ratio (g H20/kg dry air)"
else:
    lable = "unknown"

sns.heatmap(
    heatmap_year,
    cmap="coolwarm",
    cbar_kws={"label": lable})

if HOSTRADA_VAR == "tas":
    title = "Annual cycle of the air temperature"
elif HOSTRADA_VAR == "uhi":
    title = "Annual cycle of the Urban Heat Island Intensity"
elif HOSTRADA_VAR == "sfcWind":
    title = "Annual cycle of the wind speed"
elif HOSTRADA_VAR == "sfcWind_direction":
    title = "Annual cycle of the wind direction"
elif HOSTRADA_VAR == "rsds":
    title = "Annual cycle of the global radiation"
elif HOSTRADA_VAR == "clt":
    title = "Annual cycle of cloud cover"
elif HOSTRADA_VAR == "hurs":
    title = "Annual cycle of the relative humidity"
elif HOSTRADA_VAR == "tdew":
    title = "Annual cycle of the dew point temperature"
elif HOSTRADA_VAR == "mixr":
    title = "Anual water vapor mixing ratio"
else:
    title = "unknown"
    
plt.title(title)
plt.show()


## Clustered Heatmap of the HOSTRADA variable

In [ ]:
df = pd.read_csv(fn)
df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time")
df["month"] = df["time"].dt.month
df["hour"] = df["time"].dt.hour

climatology = df.pivot_table(
    values=HOSTRADA_VAR,
    index="hour",
    columns="month",
    aggfunc="mean")

plt.figure(figsize=(10,6))
plt.xlabel("Month")
plt.ylabel("Hour")

months = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

if HOSTRADA_VAR == "tas":
    lable = "Air temperature (°C)"
elif HOSTRADA_VAR == "uhi":
    lable = "Urban Heat Island Intensity (°C)"
elif HOSTRADA_VAR == "sfcWind":
    lable = "Wind speed (m/s)"
elif HOSTRADA_VAR == "sfcWind_direction":
    lable = "Wind direction (degree)"
elif HOSTRADA_VAR == "rsda":
    lable = "Global radiation (W/m2)"
elif HOSTRADA_VAR == "clt":
    lable = "Cloud cover (eighth)"
elif HOSTRADA_VAR == "hurs":
    lable = "Relative humidity (percent)"
elif HOSTRADA_VAR == "tdew":
    lable = "Dew point temperature (°C)"
elif HOSTRADA_VAR == "mixr":
    lable = "Water vapor mixing ratio (g H20/kg dry air)"
else:
    lable = "unknown"

sns.heatmap(
    climatology,
    cmap="coolwarm",
    xticklabels=months,
    cbar_kws={"label": lable}
)

if HOSTRADA_VAR == "tas":
    title = "Climatological air temperature"
elif HOSTRADA_VAR == "uhi":
    title = "Climatological Urban Heat Island Intensity"
elif HOSTRADA_VAR == "sfcWind":
    title = "Climatological wind speed"
elif HOSTRADA_VAR == "sfcWind_direction":
    title = "Climatological wind direction"
elif HOSTRADA_VAR == "rsds":
    title = "Climatological global radiation"
elif HOSTRADA_VAR == "clt":
    title = "Climatological cloud cover"
elif HOSTRADA_VAR == "clt":    
    title = "Climatological relative Feuchte"
elif HOSTRADA_VAR == "tdew":
    title = "Climatological dew point temperature"
elif HOSTRADA_VAR == "mixr":
    title = "Climatological water vapor mixing ratio"
else:
    title = "unknown"

plt.title(title)
plt.show()
